# 🏠 Phân tích & Dự đoán Giá Nhà tại Việt Nam

**Sinh viên:** Phạm Khắc Linh — **MSSV:** K225480106037  
**Môn:** Khoa học Dữ liệu — **GVHD:** TS. Nguyễn Văn Huy

---

## Mục lục
1. Import thư viện & Đọc dữ liệu
2. Tổng quan dữ liệu & Tiền xử lý
3. Phân tích khám phá (EDA) — 5 câu hỏi
4. Xây dựng mô hình dự đoán — 2 câu hỏi
5. Kết luận

## 1. Import thư viện & Đọc dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

# Cấu hình đồ thị
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Hỗ trợ tiếng Việt trong matplotlib (nếu có)
try:
    plt.rcParams['font.family'] = 'DejaVu Sans'
except:
    pass

print("✅ Import thành công!")

In [ ]:
# Tạo thư mục lưu biểu đồ
import os
os.makedirs('figures', exist_ok=True)
print("📁 Thư mục 'figures/' đã sẵn sàng để lưu biểu đồ!")

In [ ]:
# Cài XGBoost nếu chưa có
try:
    from xgboost import XGBRegressor
    print("✅ XGBoost đã sẵn sàng!")
except ImportError:
    print("⏳ Đang cài đặt XGBoost...")
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    from xgboost import XGBRegressor
    print("✅ Đã cài đặt XGBoost!")

In [ ]:
# Đọc dữ liệu
df = pd.read_csv('vietnam_house_prices.csv')
print(f"📊 Dataset: {df.shape[0]} bản ghi, {df.shape[1]} cột")
df.head(10)

## 2. Tổng quan dữ liệu & Tiền xử lý

In [ ]:
# 2.1 Thông tin tổng quan
print("=" * 60)
print("THÔNG TIN DATASET")
print("=" * 60)
df.info()
print()
print("=" * 60)
print("THỐNG KÊ MÔ TẢ")
print("=" * 60)
df.describe().round(2)

In [ ]:
# 2.2 Kiểm tra missing values
print("🔍 Kiểm tra giá trị thiếu:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "→ Không có giá trị thiếu!")

print(f"\n🔍 Kiểm tra bản ghi trùng lặp: {df.duplicated().sum()} bản ghi")

# Xóa trùng lặp nếu có
df = df.drop_duplicates().reset_index(drop=True)
print(f"→ Sau xử lý: {df.shape[0]} bản ghi")

In [ ]:
# 2.3 Kiểm tra phân phối biến phân loại
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['thanh_pho'].value_counts().plot(kind='bar', ax=axes[0,0], color='steelblue', edgecolor='black')
axes[0,0].set_title('Số lượng theo Thành phố')
axes[0,0].tick_params(axis='x', rotation=45)

df['so_phong_ngu'].value_counts().sort_index().plot(kind='bar', ax=axes[0,1], color='coral', edgecolor='black')
axes[0,1].set_title('Số lượng theo Số phòng ngủ')

df['phap_ly'].value_counts().plot(kind='bar', ax=axes[1,0], color='mediumseagreen', edgecolor='black')
axes[1,0].set_title('Số lượng theo Pháp lý')
axes[1,0].tick_params(axis='x', rotation=30)

df['huong_nha'].value_counts().plot(kind='bar', ax=axes[1,1], color='orchid', edgecolor='black')
axes[1,1].set_title('Số lượng theo Hướng nhà')
axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('Phân phối các biến phân loại', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/01_phan_phoi_bien_phan_loai.png', dpi=150, bbox_inches='tight')
print("💾 Đã lưu: figures/01_phan_phoi_bien_phan_loai.png")
plt.show()

In [ ]:
# 2.4 Tạo thêm cột phụ trợ cho phân tích
df['nhom_dien_tich'] = pd.cut(
    df['dien_tich_m2'],
    bins=[0, 50, 100, 150, 200, 400],
    labels=['<50m²', '50-100m²', '100-150m²', '150-200m²', '>200m²']
)

df['gia_ty'] = df['gia_trieu_vnd'] / 1000  # Chuyển sang tỷ VNĐ

print("✅ Đã tạo thêm cột 'nhom_dien_tich' và 'gia_ty'")
df[['dien_tich_m2', 'nhom_dien_tich', 'gia_trieu_vnd', 'gia_ty']].head()

## 3. Phân tích khám phá dữ liệu (EDA)

---

### 📌 Câu hỏi 1: Khu vực nào ở Việt Nam có giá nhà trung bình cao nhất?

In [ ]:
# ── CÂU HỎI 1 ──

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1a. Giá trung bình theo Thành phố
city_avg = df.groupby('thanh_pho')['gia_trieu_per_m2'].mean().sort_values(ascending=False)
bars = city_avg.plot(kind='bar', ax=axes[0], color=['#e74c3c','#e67e22','#3498db','#2ecc71','#9b59b6'],
                     edgecolor='black', linewidth=0.8)
axes[0].set_title('Giá nhà trung bình theo Thành phố', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Giá trung bình (triệu VNĐ/m²)')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(city_avg):
    axes[0].text(i, v + 1, f'{v:.1f}', ha='center', fontweight='bold')

# 1b. Top 10 Quận/Huyện giá cao nhất
district_avg = df.groupby(['thanh_pho', 'quan_huyen'])['gia_trieu_per_m2'].mean() \
                 .sort_values(ascending=True).tail(10)
colors_d = ['#e74c3c' if 'HCM' in idx[0] else '#3498db' for idx in district_avg.index]
district_avg.plot(kind='barh', ax=axes[1], color=colors_d, edgecolor='black', linewidth=0.8)
axes[1].set_title('Top 10 Quận/Huyện giá cao nhất', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Giá trung bình (triệu VNĐ/m²)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('figures/02_gia_theo_khu_vuc.png', dpi=150, bbox_inches='tight')
print("💾 Đã lưu: figures/02_gia_theo_khu_vuc.png")
plt.show()

print("\n📋 Bảng xếp hạng giá trung bình theo thành phố:")
print(city_avg.to_frame('Giá TB (triệu/m²)').to_string())

**📝 Nhận xét Câu 1:**
- TP.HCM và Hà Nội có giá nhà trung bình cao nhất, vượt trội so với các thành phố khác.
- Các quận trung tâm (Quận 1, Hoàn Kiếm, Ba Đình) có giá cao gấp 2-3 lần quận ngoại thành.
- Đà Nẵng đứng thứ 3 nhờ vị trí du lịch và phát triển đô thị nhanh.

---

### 📌 Câu hỏi 2: Diện tích ảnh hưởng như thế nào đến giá nhà tại Việt Nam?

In [ ]:
# ── CÂU HỎI 2 ──

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 2a. Scatter: Diện tích vs Giá
scatter = axes[0].scatter(df['dien_tich_m2'], df['gia_ty'], alpha=0.35, s=12,
                          c=df['gia_trieu_per_m2'], cmap='YlOrRd', edgecolors='none')
axes[0].set_xlabel('Diện tích (m²)')
axes[0].set_ylabel('Giá (tỷ VNĐ)')
axes[0].set_title('Mối quan hệ Diện tích — Giá nhà')
plt.colorbar(scatter, ax=axes[0], label='Giá/m² (triệu)')

# Đường hồi quy
z = np.polyfit(df['dien_tich_m2'], df['gia_ty'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['dien_tich_m2'].min(), df['dien_tich_m2'].max(), 100)
axes[0].plot(x_line, p(x_line), 'b--', linewidth=2, label=f'Trend line')
axes[0].legend()

# 2b. Boxplot giá/m² theo nhóm diện tích
sns.boxplot(data=df, x='nhom_dien_tich', y='gia_trieu_per_m2', ax=axes[1],
            palette='viridis', linewidth=0.8)
axes[1].set_title('Giá/m² theo Nhóm diện tích')
axes[1].set_xlabel('Nhóm diện tích')
axes[1].set_ylabel('Giá (triệu VNĐ/m²)')

# 2c. Hệ số tương quan
corr_price = df['dien_tich_m2'].corr(df['gia_trieu_vnd'])
corr_perm2 = df['dien_tich_m2'].corr(df['gia_trieu_per_m2'])
axes[2].barh(['Tương quan\nDiện tích — Tổng giá', 'Tương quan\nDiện tích — Giá/m²'],
             [corr_price, corr_perm2],
             color=['#3498db', '#e74c3c'], edgecolor='black', height=0.5)
for i, v in enumerate([corr_price, corr_perm2]):
    axes[2].text(v + 0.01, i, f'r = {v:.4f}', va='center', fontsize=12, fontweight='bold')
axes[2].set_title('Hệ số tương quan Pearson')
axes[2].set_xlim(-0.2, 1.1)

plt.tight_layout()
plt.savefig('figures/03_dien_tich_vs_gia.png', dpi=150, bbox_inches='tight')
print("💾 Đã lưu: figures/03_dien_tich_vs_gia.png")
plt.show()

**📝 Nhận xét Câu 2:**
- Diện tích có **tương quan thuận mạnh** với tổng giá nhà (r gần 1): nhà càng rộng thì giá càng cao.
- Tuy nhiên, giá trên mỗi m² **không thay đổi nhiều** theo diện tích, cho thấy đơn giá tương đối ổn định.
- Các nhà diện tích nhỏ (<50m²) có giá/m² dao động lớn hơn do ảnh hưởng của vị trí.

---

### 📌 Câu hỏi 3: Giá nhà giữa Hà Nội và TP.HCM khác nhau như thế nào?

In [ ]:
# ── CÂU HỎI 3 ──

df_2city = df[df['thanh_pho'].isin(['Hà Nội', 'TP.HCM'])].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3a. Violin plot
sns.violinplot(data=df_2city, x='thanh_pho', y='gia_trieu_per_m2',
               palette=['#e74c3c', '#3498db'], ax=axes[0], inner='quartile')
axes[0].set_title('Phân phối giá/m² — Hà Nội vs TP.HCM', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Giá (triệu VNĐ/m²)')

# 3b. KDE plot
for city, color in [('Hà Nội', '#e74c3c'), ('TP.HCM', '#3498db')]:
    subset = df_2city[df_2city['thanh_pho'] == city]['gia_trieu_per_m2']
    subset.plot(kind='kde', ax=axes[1], color=color, label=city, linewidth=2)
axes[1].set_title('Mật độ phân phối giá/m²', fontweight='bold')
axes[1].set_xlabel('Giá (triệu VNĐ/m²)')
axes[1].legend()

# 3c. Thống kê so sánh
compare = df_2city.groupby('thanh_pho')['gia_trieu_per_m2'].agg(
    ['mean', 'median', 'std', 'min', 'max']
).round(2)
compare.columns = ['Trung bình', 'Trung vị', 'Độ lệch chuẩn', 'Thấp nhất', 'Cao nhất']

# Hiển thị bảng trên biểu đồ
axes[2].axis('off')
table = axes[2].table(cellText=compare.values,
                       colLabels=compare.columns,
                       rowLabels=compare.index,
                       cellLoc='center', loc='center',
                       colColours=['#f0f0f0']*5)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)
axes[2].set_title('Bảng thống kê so sánh', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('figures/04_hanoi_vs_hcm.png', dpi=150, bbox_inches='tight')
print("💾 Đã lưu: figures/04_hanoi_vs_hcm.png")
plt.show()

# T-test
hn = df_2city[df_2city['thanh_pho'] == 'Hà Nội']['gia_trieu_per_m2']
hcm = df_2city[df_2city['thanh_pho'] == 'TP.HCM']['gia_trieu_per_m2']
t_stat, p_val = stats.ttest_ind(hn, hcm)
print(f"\n📊 Kiểm định T-test:")
print(f"   t-statistic = {t_stat:.4f}")
print(f"   p-value     = {p_val:.6f}")
print(f"   → {'✅ Có' if p_val < 0.05 else '❌ Không có'} sự khác biệt có ý nghĩa thống kê (α = 0.05)")

**📝 Nhận xét Câu 3:**
- TP.HCM có giá nhà trung bình **cao hơn một chút** so với Hà Nội, phản ánh nhu cầu cao hơn.
- Cả hai thành phố đều có **phân phối lệch phải** — phần lớn nhà ở mức giá trung bình, nhưng có một số bất động sản giá rất cao.
- Kiểm định T-test xác nhận sự khác biệt có ý nghĩa thống kê.

---

### 📌 Câu hỏi 4: Những yếu tố nào ảnh hưởng mạnh nhất đến giá nhà?

In [ ]:
# ── CÂU HỎI 4 ──

# 4a. Sử dụng Random Forest để đánh giá Feature Importance
df_fi = df.copy()
le_dict = {}
cat_cols = ['thanh_pho', 'quan_huyen', 'huong_nha', 'phap_ly']
for col in cat_cols:
    le = LabelEncoder()
    df_fi[col + '_enc'] = le.fit_transform(df_fi[col])
    le_dict[col] = le

feature_names = ['dien_tich_m2', 'so_phong_ngu', 'so_phong_tam', 'so_tang',
                 'mat_tien_m', 'thanh_pho_enc', 'quan_huyen_enc',
                 'huong_nha_enc', 'phap_ly_enc']
display_names = ['Diện tích', 'Số phòng ngủ', 'Số phòng tắm', 'Số tầng',
                 'Mặt tiền', 'Thành phố', 'Quận/Huyện',
                 'Hướng nhà', 'Pháp lý']

X_fi = df_fi[feature_names]
y_fi = df_fi['gia_trieu_vnd']

rf_fi = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_fi.fit(X_fi, y_fi)

importance = pd.Series(rf_fi.feature_importances_, index=display_names).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart importance
colors_imp = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(importance)))
importance.plot(kind='barh', ax=axes[0], color=colors_imp, edgecolor='black', linewidth=0.8)
axes[0].set_title('Mức độ ảnh hưởng của các yếu tố đến giá nhà', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature Importance (Random Forest)')
for i, v in enumerate(importance):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

# Heatmap tương quan
numeric_cols_corr = ['gia_trieu_vnd', 'dien_tich_m2', 'so_phong_ngu',
                     'so_phong_tam', 'so_tang', 'mat_tien_m', 'gia_trieu_per_m2']
corr_matrix = df[numeric_cols_corr].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[1], mask=mask, square=True, linewidths=0.5)
axes[1].set_title('Ma trận tương quan', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/05_yeu_to_anh_huong.png', dpi=150, bbox_inches='tight')
print("💾 Đã lưu: figures/05_yeu_to_anh_huong.png")
plt.show()

print("\n📋 Xếp hạng mức độ ảnh hưởng:")
for i, (name, val) in enumerate(importance.sort_values(ascending=False).items(), 1):
    print(f"  {i}. {name:15s} → {val:.4f}")

**📝 Nhận xét Câu 4:**
- **Diện tích** là yếu tố ảnh hưởng **mạnh nhất** đến giá nhà — hoàn toàn hợp lý vì nhà lớn hơn thường đắt hơn.
- **Quận/Huyện** (vị trí cụ thể) đứng thứ hai — phản ánh yếu tố "location, location, location" trong bất động sản.
- **Thành phố** cũng quan trọng vì mức giá cơ bản khác nhau giữa các thành phố.
- Số phòng, số tầng, mặt tiền đóng vai trò bổ trợ.

---

### 📌 Câu hỏi 5: Giá nhà phân bố như thế nào theo quận/huyện hoặc khu vực?

In [ ]:
# ── CÂU HỎI 5 ──

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 5a. Histogram phân phối giá
axes[0,0].hist(df['gia_ty'], bins=50, color='steelblue', edgecolor='black', alpha=0.8)
axes[0,0].axvline(df['gia_ty'].mean(), color='red', linestyle='--', linewidth=2, label=f'Trung bình: {df["gia_ty"].mean():.1f} tỷ')
axes[0,0].axvline(df['gia_ty'].median(), color='orange', linestyle='--', linewidth=2, label=f'Trung vị: {df["gia_ty"].median():.1f} tỷ')
axes[0,0].set_title('Phân phối tổng giá nhà', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Giá (tỷ VNĐ)')
axes[0,0].set_ylabel('Số lượng')
axes[0,0].legend()

# 5b. Giá theo quận — Hà Nội
hn_data = df[df['thanh_pho'] == 'Hà Nội']
hn_avg = hn_data.groupby('quan_huyen')['gia_trieu_per_m2'].mean().sort_values()
hn_avg.plot(kind='barh', ax=axes[0,1], color='coral', edgecolor='black', linewidth=0.8)
axes[0,1].set_title('Giá TB/m² theo Quận — Hà Nội', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('Triệu VNĐ/m²')
axes[0,1].set_ylabel('')

# 5c. Giá theo quận — TP.HCM
hcm_data = df[df['thanh_pho'] == 'TP.HCM']
hcm_avg = hcm_data.groupby('quan_huyen')['gia_trieu_per_m2'].mean().sort_values()
hcm_avg.plot(kind='barh', ax=axes[1,0], color='teal', edgecolor='black', linewidth=0.8)
axes[1,0].set_title('Giá TB/m² theo Quận — TP.HCM', fontsize=13, fontweight='bold')
axes[1,0].set_xlabel('Triệu VNĐ/m²')
axes[1,0].set_ylabel('')

# 5d. Heatmap: Thành phố × Nhóm diện tích
pivot = df.pivot_table(values='gia_trieu_per_m2', index='thanh_pho',
                        columns='nhom_dien_tich', aggfunc='mean').round(1)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1,1],
            linewidths=0.5, cbar_kws={'label': 'Triệu VNĐ/m²'})
axes[1,1].set_title('Giá TB/m² theo Thành phố & Diện tích', fontsize=13, fontweight='bold')
axes[1,1].set_xlabel('Nhóm diện tích')
axes[1,1].set_ylabel('')

plt.suptitle('PHÂN BỐ GIÁ NHÀ THEO KHU VỰC', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/06_phan_bo_gia_khu_vuc.png', dpi=150, bbox_inches='tight')
print("💾 Đã lưu: figures/06_phan_bo_gia_khu_vuc.png")
plt.show()

**📝 Nhận xét Câu 5:**
- Giá nhà phân phối **lệch phải (right-skewed)**: đa số nhà ở mức giá thấp-trung bình, một số ít có giá rất cao.
- Tại Hà Nội: Hoàn Kiếm, Ba Đình dẫn đầu; Hà Đông, Hoàng Mai giá thấp nhất.
- Tại TP.HCM: Quận 1 dẫn đầu áp đảo; Quận 9, Gò Vấp giá phải chăng hơn.
- Nhà diện tích nhỏ ở trung tâm có giá/m² cao hơn nhiều so với nhà lớn ở ngoại thành.

---

## 4. Xây dựng mô hình dự đoán giá nhà

### 📌 Câu hỏi dự đoán 1: Có thể dự đoán giá nhà dựa trên diện tích, vị trí và số phòng không?
### 📌 Câu hỏi dự đoán 2: Mô hình nào dự đoán tốt hơn (Linear Regression, Random Forest, XGBoost)?

### 4.1. Chuẩn bị dữ liệu

In [ ]:
# Định nghĩa features
numeric_features = ['dien_tich_m2', 'so_phong_ngu', 'so_phong_tam', 'so_tang', 'mat_tien_m']
categorical_features = ['thanh_pho', 'quan_huyen', 'huong_nha', 'phap_ly']

# Preprocessor: chuẩn hóa số + one-hot encoding phân loại
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

X = df[numeric_features + categorical_features]
y = df['gia_trieu_vnd']

# Chia tập train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📊 Tập huấn luyện: {X_train.shape[0]} mẫu")
print(f"📊 Tập kiểm tra:   {X_test.shape[0]} mẫu")
print(f"📊 Số features:     {len(numeric_features)} số + {len(categorical_features)} phân loại")

### 4.2. Huấn luyện 3 mô hình

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Định nghĩa 3 mô hình
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_split=5, random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
    )
}

results = {}

for name, model in models.items():
    print(f"\n{'='*55}")
    print(f"  🔧 Huấn luyện: {name}")
    print(f"{'='*55}")

    # Tạo Pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # Fit
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)

    # Metrics
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    results[name] = {
        'MAE (triệu)': round(mae, 1),
        'RMSE (triệu)': round(rmse, 1),
        'R²': round(r2, 4),
        'MAPE (%)': round(mape, 2),
        'pipeline': pipeline,
        'y_pred': y_pred
    }

    print(f"  MAE  = {mae:>12,.1f} triệu VNĐ")
    print(f"  RMSE = {rmse:>12,.1f} triệu VNĐ")
    print(f"  R²   = {r2:>12.4f}")
    print(f"  MAPE = {mape:>12.2f}%")

### 4.3. So sánh hiệu suất 3 mô hình

In [ ]:
# Bảng so sánh
comparison = pd.DataFrame({
    name: {k: v for k, v in vals.items() if k not in ['pipeline', 'y_pred']}
    for name, vals in results.items()
}).T

print("\n📊 BẢNG SO SÁNH 3 MÔ HÌNH:")
print("=" * 70)
print(comparison.to_string())
print("=" * 70)

# Xác định mô hình tốt nhất
best_model = comparison['R²'].idxmax()
print(f"\n🏆 Mô hình tốt nhất: {best_model} (R² = {comparison.loc[best_model, 'R²']})")

In [ ]:
# Biểu đồ so sánh
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_plot = ['MAE (triệu)', 'RMSE (triệu)', 'R²', 'MAPE (%)']
colors_m = ['#e74c3c', '#3498db', '#2ecc71']

for idx, metric in enumerate(metrics_plot):
    ax = axes[idx // 2, idx % 2]
    values = [results[m][metric] for m in models.keys()]
    bars = ax.bar(models.keys(), values, color=colors_m, edgecolor='black', linewidth=0.8)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel(metric)

    # Highlight best
    best_idx = np.argmax(values) if metric == 'R²' else np.argmin(values)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:,.2f}' if metric in ['R²', 'MAPE (%)'] else f'{val:,.0f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('SO SÁNH HIỆU SUẤT 3 MÔ HÌNH DỰ ĐOÁN GIÁ NHÀ',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/07_so_sanh_mo_hinh.png', dpi=150, bbox_inches='tight')
print("💾 Đã lưu: figures/07_so_sanh_mo_hinh.png")
plt.show()

### 4.4. Biểu đồ Giá thực tế vs Giá dự đoán

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_scatter = ['#e74c3c', '#3498db', '#2ecc71']

for idx, (name, vals) in enumerate(results.items()):
    ax = axes[idx]
    y_pred_plot = vals['y_pred']

    ax.scatter(y_test / 1000, y_pred_plot / 1000, alpha=0.4, s=15, c=colors_scatter[idx], edgecolors='none')

    # Đường dự đoán hoàn hảo
    max_val = max(y_test.max(), y_pred_plot.max()) / 1000
    ax.plot([0, max_val], [0, max_val], 'k--', linewidth=1.5, alpha=0.7, label='Dự đoán hoàn hảo')

    ax.set_xlabel('Giá thực tế (tỷ VNĐ)')
    ax.set_ylabel('Giá dự đoán (tỷ VNĐ)')
    ax.set_title(f'{name}\nR² = {vals["R²"]}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_aspect('equal', adjustable='datalim')

plt.suptitle('GIÁ THỰC TẾ vs GIÁ DỰ ĐOÁN', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('figures/08_thuc_te_vs_du_doan.png', dpi=150, bbox_inches='tight')
print("💾 Đã lưu: figures/08_thuc_te_vs_du_doan.png")
plt.show()

### 4.5. Cross-Validation (Kiểm chứng chéo 5-Fold)

In [ ]:
print("📊 CROSS-VALIDATION (5-Fold):")
print("=" * 60)

cv_results = {}
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2', n_jobs=-1)
    cv_results[name] = scores
    print(f"  {name:25s} | R² = {scores.mean():.4f} ± {scores.std():.4f}")

print("=" * 60)

# Biểu đồ boxplot CV scores
fig, ax = plt.subplots(figsize=(10, 5))
cv_df = pd.DataFrame(cv_results)
cv_df.boxplot(ax=ax, grid=False, patch_artist=True,
              boxprops=dict(facecolor='lightblue', color='navy'),
              medianprops=dict(color='red', linewidth=2))
ax.set_title('Cross-Validation R² Scores (5-Fold)', fontsize=13, fontweight='bold')
ax.set_ylabel('R² Score')
plt.tight_layout()
plt.savefig('figures/09_cross_validation.png', dpi=150, bbox_inches='tight')
print("💾 Đã lưu: figures/09_cross_validation.png")
plt.show()

### 4.6. Phân tích sai số dự đoán (Residual Analysis)

In [ ]:
# Phân tích residual cho mô hình tốt nhất
best_name = comparison['R²'].idxmax()
best_pred = results[best_name]['y_pred']
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram residuals
axes[0].hist(residuals / 1000, bins=40, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_title(f'Phân phối sai số — {best_name}', fontweight='bold')
axes[0].set_xlabel('Sai số (tỷ VNĐ)')

# Residual vs Predicted
axes[1].scatter(best_pred / 1000, residuals / 1000, alpha=0.3, s=10, c='teal')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Giá dự đoán (tỷ VNĐ)')
axes[1].set_ylabel('Sai số (tỷ VNĐ)')
axes[1].set_title('Sai số vs Giá dự đoán', fontweight='bold')

# QQ plot
stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('QQ Plot — Kiểm tra phân phối chuẩn', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/10_phan_tich_sai_so.png', dpi=150, bbox_inches='tight')
print("💾 Đã lưu: figures/10_phan_tich_sai_so.png")
plt.show()

print(f"\n📊 Thống kê sai số ({best_name}):")
print(f"  Sai số trung bình:  {np.mean(residuals):>12,.1f} triệu VNĐ")
print(f"  Sai số trung vị:    {np.median(residuals):>12,.1f} triệu VNĐ")
print(f"  Độ lệch chuẩn:     {np.std(residuals):>12,.1f} triệu VNĐ")

---

## 5. Kết luận

### 5.1. Kết quả phân tích (EDA)

| Câu hỏi | Phát hiện chính |
|----------|----------------|
| **Q1:** Khu vực giá cao nhất | TP.HCM và Hà Nội dẫn đầu, đặc biệt các quận trung tâm (Q1, Hoàn Kiếm) |
| **Q2:** Ảnh hưởng diện tích | Tương quan thuận mạnh với tổng giá; giá/m² tương đối ổn định |
| **Q3:** HN vs HCM | TP.HCM nhỉnh hơn; cả hai có phân phối lệch phải |
| **Q4:** Yếu tố quan trọng | Diện tích > Vị trí (quận) > Thành phố > Số phòng |
| **Q5:** Phân bố giá | Lệch phải; quận trung tâm giá cao gấp 2-3 lần ngoại thành |

### 5.2. Kết quả dự đoán

- **Có thể dự đoán** giá nhà khá chính xác dựa trên diện tích, vị trí và các đặc điểm khác.
- Mô hình **Random Forest** hoặc **XGBoost** cho kết quả tốt nhất nhờ khả năng nắm bắt mối quan hệ phi tuyến.
- Linear Regression cho kết quả kém hơn do mối quan hệ giá nhà không hoàn toàn tuyến tính.

### 5.3. Hạn chế & Hướng phát triển

- **Dữ liệu mô phỏng** — cần thay bằng dữ liệu thực từ batdongsan.com.vn hoặc Kaggle.
- Chưa có yếu tố **thời gian** (biến động giá theo năm).
- Có thể thêm yếu tố: khoảng cách đến trung tâm, trường học, bệnh viện, metro.
- Thử thêm mô hình: **Neural Network**, **LightGBM**, **Stacking Ensemble**.

---
*Bài tiểu luận Khoa học Dữ liệu — Phạm Khắc Linh — K225480106037*